In [80]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
from time import time

from sklearn.pipeline import (
    Pipeline, make_pipeline
)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, train_test_split,
    KFold, StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier
)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import optuna

In [81]:
SEED = 67
WORK_DIR = Path(os.getcwd()).parent
DATA_DIR = WORK_DIR / 'data' / 'interim'

In [82]:
df = pd.read_csv(DATA_DIR / 'interim_hotel_bookings.csv', encoding='utf-8', parse_dates=['reservation_status_date']).drop_duplicates()
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,0,0,0,Transient,75.0,0,0,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304,0,0,Transient,75.0,0,0,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240,0,0,Transient,98.0,0,1,2015-07-03


In [83]:
y = df['is_canceled']
X = df.drop(columns=['is_canceled', 'reservation_status_date', 'booking_changes', 'days_in_waiting_list', 'assigned_room_type'])
X.shape, y.shape

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=SEED
)

X_sample, _, y_sample, _ = train_test_split(
    X_train, y_train, stratify=y_train, train_size=(X_train.shape[0]//8), test_size=0.2, random_state=SEED
)

In [84]:
numeric = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies',
    'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces',
    'total_of_special_requests'
]
categorical = X.columns.difference(numeric).tolist()

def cust_pipeline(model) -> Pipeline:
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('scale', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50, sparse_output=True))
    ])

    preprocessing = ColumnTransformer([
        ('numeric', num_pipe, numeric),
        ('categorical', cat_pipe, categorical)
    ])

    return Pipeline([('preprocessing', preprocessing), ('model', model)])

def models_params():
    return {
        'LogisticRegression': [LogisticRegression(max_iter=1000, random_state=SEED), 
                               {'model__C': [0.01, 0.1, 1, 10, 100]}],
        'KNeighborsClassifier': [KNeighborsClassifier(n_jobs=-1), 
                                 {'model__weights': ['uniform', 'distance'], 
                                  'model__n_neighbors': [15, 30, 50]}],
        'SVM': [SVC(probability=True, random_state=SEED), 
                {'model__C': [0.1, 1, 10], 
                 'model__kernel': ['poly', 'rbf']}],
        'MLPClassifier': [MLPClassifier(learning_rate='adaptive', max_iter=1000, early_stopping=True, random_state=SEED), 
                          {'model__hidden_layer_sizes': [(32,), (64,), (16, 32), (64, 32), (16, 16, 16)]}],
        'RandomForestClassifier': [RandomForestClassifier(n_jobs=-1, random_state=SEED),
                                   {'model__n_estimators': [100, 150, 250],
                                    'model__max_depth': [10, 16, 24]}],
        'CatBoostClassifier': [CatBoostClassifier(learning_rate=0.07, eval_metric='F1', iterations=400, verbose=False, thread_count=-1, random_state=SEED),
                               {'model__depth': [5, 8, 12]}],
        'LGBMClassifier': [LGBMClassifier(learning_rate=0.06, verbosity=-1, random_state=SEED, n_jobs=-1), 
                           {'model__n_estimators': [100, 150, 250],
                            'model__reg_lambda': [0.2, 0.4, 0.8],
                            'model__boosting_type': ['gbdt', 'dart']}],
        'AdaBoostClassifier': [AdaBoostClassifier(learning_rate=1.2, random_state=SEED),
                            {'model__n_estimators': [50, 100, 150, 250]}]
    }

In [85]:
(df['is_canceled'].value_counts() / df.shape[0] * 100).round(2)

is_canceled
0    72.51
1    27.49
Name: count, dtype: float64

In [86]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date
count,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,...,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394.000000,87394
mean,0.274882,79.890610,2016.210300,26.838353,15.815479,1.005252,2.625443,1.875804,0.138637,0.010825,...,0.030414,0.183994,0.271586,81.002300,11.017061,0.749582,106.337506,0.084228,0.698583,2016-08-31 03:46:33.610545
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.380000,0.000000,0.000000,2014-10-17 00:00:00
25%,0.000000,11.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,72.000000,0.000000,0.000000,2016-03-18 00:00:00
50%,0.000000,49.000000,2016.000000,27.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,98.100000,0.000000,0.000000,2016-09-08 00:00:00
75%,1.000000,125.000000,2017.000000,37.000000,23.000000,2.000000,4.000000,2.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,234.000000,0.000000,0.000000,134.000000,0.000000,1.000000,2017-03-05 00:00:00
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,...,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000,2017-09-14 00:00:00
std,0.446457,86.053163,0.686109,13.674513,8.835121,1.031928,2.053581,0.626500,0.455876,0.113598,...,0.369149,1.731913,0.727245,109.945532,54.048077,10.015845,55.014426,0.281536,0.831949,NaN


In [87]:
fold_results = []
results = []
test_probabilities = {}
best_models = {}

def get_metrics(y_true, probability, threshold=0.5):
    predicted = (probability >= threshold).astype('int')

    return {
        'test_accuracy': accuracy_score(y_true, predicted),
        'test_precision': precision_score(y_true, predicted, zero_division=0),
        'test_recall': recall_score(y_true, predicted, zero_division=0),
        'test_f1': f1_score(y_true, predicted, zero_division=0),
        'test_roc_auc': roc_auc_score(y_true, probability),
        'test_pr_auc': average_precision_score(y_true, probability)
    }

def model_result(name: str) -> pd.DataFrame:
    model, param_grid = models_params()[name]
    cv_name, cv = ('StratifiedKFold', StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED))
    search = RandomizedSearchCV(
        cust_pipeline(model=model),
        param_distributions=param_grid,
        n_iter=4,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        refit=True,
        return_train_score=True
    )
    search.fit(X_sample, y_sample)
    i = search.best_index_
    fold_results.append(
        {
            'model': name, 'cv': cv_name,
            'best_train_score': search.cv_results_['mean_train_score'][i],
            'best_test_score': search.cv_results_['mean_test_score'][i]
        }
    )
    p = search.predict_proba(X_test)[:, 1]
    test_probabilities[name] = p
    best_models[name] = search.best_estimator_
    results.append(
        {
            'model': name,
            'f1_best_score': search.best_score_,
            **get_metrics(y_test, p, threshold=0.42)
        }
    )  
    return pd.DataFrame(results)

In [119]:
def model_objective(trial):

    name = trial.suggest_categorical(
        'model',
        [
            'LogisticRegression',
            'KNeighborsClassifier',
            'SVM',
            'MLPClassifier',
            'RandomForestClassifier',
            'CatBoostClassifier',
            'LGBMClassifier',
            'AdaBoostClassifier'
        ]
    )

    if name == 'LogisticRegression':
        return cust_pipeline(LogisticRegression(
            C=trial.suggest_float('C', 1e-3, 100, log=True),
            max_iter=1000,
            random_state=SEED,
            n_jobs=-1
        ))

    if name == 'KNeighborsClassifier':
        return cust_pipeline(KNeighborsClassifier(
            weights=trial.suggest_categorical('weights', [
                'uniform', 'distance'
            ]),
            n_neighbors=trial.suggest_int('n_neighbors', 10, 150),
            n_jobs=-1
        ))

    if name == 'SVM':
        return cust_pipeline(SVC(
            C=trial.suggest_float('C', 1e-2, 100, log=True),
            kernel=trial.suggest_categorical('kernel', ['rbf', 'poly', 'linear']),
            max_iter=1000,
            random_state=SEED
        ))

    if name == 'MLPClassifier':
        return cust_pipeline(MLPClassifier(
            hidden_layer_sizes=trial.suggest_categorical('hidden_layer_sizes', [
                (32,), (64,), (16, 32), (64, 32), (16, 16, 16)
            ]),
            activation=trial.suggest_categorical('activation', [
                'relu', 'tanh', 'logistic'
            ]),
            alpha=trial.suggest_float('alpha', 1e-6, 1e-2, log=True),
            learning_rate='adaptive',
            early_stopping=True,
            random_state=SEED
        ))

    if name == 'RandomForestClassifier':
        return cust_pipeline(RandomForestClassifier(
            criterion=trial.suggest_categorical('criterion', [
                'gini', 'log_loss', 'entropy'
            ]),
            max_depth=trial.suggest_int('max_depth', 4, 12),
            n_estimators=trial.suggest_int('n_estimators', 50, 300),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 2, 50),
            random_state=SEED,
            n_jobs=-1
        ))

    if name == 'CatBoostClassifier':
        return cust_pipeline(CatBoostClassifier(
            iterations=trial.suggest_int('iterations', 200, 600),
            depth=trial.suggest_int('depth', 4, 12),
            learning_rate=trial.suggest_float(
                'learning_rate', 0.01, 0.2, log=True
            ),
            verbose=False,
            random_seed=SEED,
            thread_count=-1,
        ))

    if name == 'LightGBMClassifier':
        return cust_pipeline(LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 100, 500),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            learning_rate=trial.suggest_float(
                'learning_rate', 0.01, 0.2, log=True
            ),
            num_leaves=trial.suggest_int('num_leaves', 15, 80),
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ))

    return cust_pipeline(AdaBoostClassifier(
        n_estimators=trial.suggest_int('n_estimators', 50, 400),
        learning_rate=trial.suggest_float(
            'learning_rate', 0.01, 2.0, log=True
        ),
        random_state=SEED,
    ))

def objective(trial):

    scores = cross_val_score(
        model_objective(trial),
        X_sample,
        y_sample,
        scoring='average_precision',
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
        n_jobs=-1
    )
    return scores.mean()

In [124]:
def final_model(params: dict):
    name = params['model']

    if name == 'LogisticRegression':
        return cust_pipeline(LogisticRegression(
            C=params['C'],
            max_iter=1000,
            random_state=SEED,
            n_jobs=-1
        ))
    
    if name == 'KNeighborsClassifier':
        return cust_pipeline(KNeighborsClassifier(
            weights=params['weights'],
            n_neighbors=params['n_neighbors'],
            n_jobs=-1
        ))
    
    if name == 'SVM':
        return cust_pipeline(SVC(
            C=params['C'],
            kernel=params['kernel'],
            max_iter=1000,
            random_state=SEED
        ))
    
    if name == 'MLPClassifier':
        return cust_pipeline(MLPClassifier(
            hidden_layer_sizes=params['hidden_layer_sizes'],
            activation=params['activation'],
            alpha=params['alpha'],
            learning_rate='adaptive',
            early_stopping=True,
            random_state=SEED
        ))
    
    if name == 'RandomForestClassifier':
        return cust_pipeline(RandomForestClassifier(
            criterion=params['criterion'],
            max_depth=params['max_depth'],
            n_estimators=params['n_estimators'],
            min_samples_leaf=params['min_samples_leaf'],
            random_state=SEED,
            n_jobs=-1
        ))
    
    if name == 'CatBoostClassifier':
        return cust_pipeline(CatBoostClassifier(
            iterations=params['iterations'],
            depth=params['depth'],
            learning_rate=params['learning_rate'],
            verbose=False,
            random_seed=SEED,
            thread_count=1,
        ))
    
    if name == 'LightGBMCLassifier':
        return cust_pipeline(LGBMClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            num_leaves=params['num_leaves'],
            random_state=SEED,
            n_jobs=1,
            verbosity=-1,
        ))

    return cust_pipeline(AdaBoostClassifier(
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        random_state=SEED,
    ))

In [90]:
randGS_time0 = time()

In [91]:
best_log_reg = model_result('LogisticRegression')

In [92]:
best_log_reg

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298


In [93]:
best_KNN = model_result('KNeighborsClassifier')

In [94]:
best_KNN

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260


In [95]:
best_svm = model_result('SVM')

c:\Users\Maksim-Book\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [96]:
best_svm

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260
2,SVM,0.607209,0.808570,0.709807,0.513632,0.595991,0.849573,0.683453


In [97]:
best_mlp = model_result('MLPClassifier')

In [98]:
best_mlp

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260
2,SVM,0.607209,0.808570,0.709807,0.513632,0.595991,0.849573,0.683453
3,MLPClassifier,0.612080,0.807655,0.648916,0.654318,0.651606,0.871975,0.720176


In [99]:
best_forest = model_result('RandomForestClassifier')

In [100]:
best_forest

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260
2,SVM,0.607209,0.808570,0.709807,0.513632,0.595991,0.849573,0.683453
3,MLPClassifier,0.612080,0.807655,0.648916,0.654318,0.651606,0.871975,0.720176
4,RandomForestClassifier,0.619274,0.832370,0.720744,0.637045,0.676315,0.890190,0.759684


In [101]:
best_catboost = model_result('CatBoostClassifier')

c:\Users\Maksim-Book\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [102]:
best_catboost

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260
2,SVM,0.607209,0.808570,0.709807,0.513632,0.595991,0.849573,0.683453
3,MLPClassifier,0.612080,0.807655,0.648916,0.654318,0.651606,0.871975,0.720176
4,RandomForestClassifier,0.619274,0.832370,0.720744,0.637045,0.676315,0.890190,0.759684
5,CatBoostClassifier,0.655684,0.831970,0.694746,0.693444,0.694094,0.895963,0.767924


In [103]:
best_lgbm = model_result('LGBMClassifier')

In [104]:
best_lgbm

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260
2,SVM,0.607209,0.808570,0.709807,0.513632,0.595991,0.849573,0.683453
3,MLPClassifier,0.612080,0.807655,0.648916,0.654318,0.651606,0.871975,0.720176
4,RandomForestClassifier,0.619274,0.832370,0.720744,0.637045,0.676315,0.890190,0.759684
5,CatBoostClassifier,0.655684,0.831970,0.694746,0.693444,0.694094,0.895963,0.767924
6,LGBMClassifier,0.663928,0.830196,0.684772,0.708429,0.696399,0.895403,0.764986


In [105]:
best_ada = model_result('AdaBoostClassifier')
randGS_time1 = time()

In [106]:
best_ada

,model,f1_best_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,LogisticRegression,0.550989,0.792093,0.628484,0.596046,0.611835,0.846172,0.665298
1,KNeighborsClassifier,0.538821,0.784599,0.612945,0.587305,0.599851,0.824022,0.657260
2,SVM,0.607209,0.808570,0.709807,0.513632,0.595991,0.849573,0.683453
3,MLPClassifier,0.612080,0.807655,0.648916,0.654318,0.651606,0.871975,0.720176
4,RandomForestClassifier,0.619274,0.832370,0.720744,0.637045,0.676315,0.890190,0.759684
5,CatBoostClassifier,0.655684,0.831970,0.694746,0.693444,0.694094,0.895963,0.767924
6,LGBMClassifier,0.663928,0.830196,0.684772,0.708429,0.696399,0.895403,0.764986
7,AdaBoostClassifier,0.579664,0.509812,0.357818,0.985432,0.525003,0.857760,0.675088


In [111]:
optuna_time0 = time()

study = optuna.create_study(sampler=optuna.samplers.TPESampler(), direction='maximize')
study.optimize(objective, n_trials=50)

optuna_time1 = time()

[I 2026-11-27 00:40:49,278] A new study created in memory with name: no-name-40c05a7f-28ff-41c3-aa45-5810bdb4030f
[I 2026-11-27 00:40:53,680] Trial 0 finished with value: 0.6393695181618665 and parameters: {'model': 'LogisticRegression', 'C': 14.260071961068986}. Best is trial 0 with value: 0.6393695181618665.
[I 2026-11-27 00:40:56,779] Trial 1 finished with value: 0.6307876098901971 and parameters: {'model': 'LogisticRegression', 'C': 0.003979092530680461}. Best is trial 0 with value: 0.6393695181618665.
[I 2026-11-27 00:41:01,479] Trial 2 finished with value: 0.3542986510857922 and parameters: {'model': 'SVM', 'C': 78.00757634419942, 'kernel': 'linear'}. Best is trial 0 with value: 0.6393695181618665.
[I 2026-11-27 00:41:05,575] Trial 3 finished with value: 0.3524750965107886 and parameters: {'model': 'SVM', 'C': 0.598990954693117, 'kernel': 'linear'}. Best is trial 0 with value: 0.6393695181618665.
[I 2026-11-27 00:41:06,835] Trial 4 finished with value: 0.608349764717197 and param

In [112]:
print(f'Суммарное время обучения 8 моделей при помощи RandomizedGridSearchCV: {np.round(randGS_time1 - randGS_time0, 2)}\n')
print(f'Суммарное время обучения 8 моделей при помощи Optuna: {np.round(optuna_time1 - optuna_time0, 2)}')

Суммарное время обучения 8 моделей при помощи RandomizedGridSearchCV: 184.72

Суммарное время обучения 8 моделей при помощи Optuna: 430.99


In [113]:
study.best_value, study.best_params, study.best_trial

(0.7562695091855876,
 {'model': 'CatBoostClassifier',
  'iteration': 322,
  'depth': 8,
  'learning_rate': 0.031720570703863665},
 FrozenTrial(number=31, state=<TrialState.COMPLETE: 1>, values=[0.7562695091855876], datetime_start=datetime.datetime(2026, 11, 27, 0, 46, 58, 121059), datetime_complete=datetime.datetime(2026, 11, 27, 0, 47, 2, 341830), params={'model': 'CatBoostClassifier', 'iteration': 322, 'depth': 8, 'learning_rate': 0.031720570703863665}, user_attrs={}, system_attrs={'tpe:relative_params:0': '{"model": "CatBoostClassifier"}'}, intermediate_values={}, distributions={'model': CategoricalDistribution(choices=('LogisticRegression', 'KNeighborsClassifier', 'SVM', 'MLPClassifier', 'RandomForestClassifier', 'CatBoostClassifier', 'LGBMClassifier', 'AdaBoostClassifier')), 'iteration': IntDistribution(high=600, log=False, low=200, step=1), 'depth': IntDistribution(high=12, log=False, low=4, step=1), 'learning_rate': FloatDistribution(high=0.2, log=True, low=0.01, step=None)}, tr

In [114]:
best_trial = study.best_trial
best_model_name = study.best_params['model']
best_trial, best_model_name

(FrozenTrial(number=31, state=<TrialState.COMPLETE: 1>, values=[0.7562695091855876], datetime_start=datetime.datetime(2026, 11, 27, 0, 46, 58, 121059), datetime_complete=datetime.datetime(2026, 11, 27, 0, 47, 2, 341830), params={'model': 'CatBoostClassifier', 'iteration': 322, 'depth': 8, 'learning_rate': 0.031720570703863665}, user_attrs={}, system_attrs={'tpe:relative_params:0': '{"model": "CatBoostClassifier"}'}, intermediate_values={}, distributions={'model': CategoricalDistribution(choices=('LogisticRegression', 'KNeighborsClassifier', 'SVM', 'MLPClassifier', 'RandomForestClassifier', 'CatBoostClassifier', 'LGBMClassifier', 'AdaBoostClassifier')), 'iteration': IntDistribution(high=600, log=False, low=200, step=1), 'depth': IntDistribution(high=12, log=False, low=4, step=1), 'learning_rate': FloatDistribution(high=0.2, log=True, low=0.01, step=None)}, trial_id=31, value=None),
 'CatBoostClassifier')

In [115]:
best_trial.params

{'model': 'CatBoostClassifier',
 'iteration': 322,
 'depth': 8,
 'learning_rate': 0.031720570703863665}

In [121]:
best_model = final_model(best_trial.params)
best_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](26,)","['hotel','lead_time','arrival_date_year',...,'adr', 'required_car_parking_spaces','total_of_special_requests']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,26
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop

In [125]:
opt_df = pd.DataFrame(get_metrics(y_true=y_test, probability=best_model.predict_proba(X_test)[:, 1]), index=('CatBoostClassifier',))
opt_df

,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
CatBoostClassifier,0.839865,0.747166,0.631009,0.684193,0.905135,0.78662


In [126]:
rgscv_best_model = best_models['CatBoostClassifier'].fit(X_train, y_train)
pd.DataFrame(get_metrics(y_true=y_test, probability=rgscv_best_model.predict_proba(X_test)[:, 1]), index=('CatBoostClassifier',))

,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
CatBoostClassifier,0.838435,0.740121,0.63538,0.683763,0.906132,0.788528
